# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The .metadata object provides an easy-to-use interface
print("Dataset Title:", dataset.metadata.name)
print("Description  :", dataset.metadata.description)
print("Published on :", dataset.metadata.date_published)
print("Version      :", dataset.metadata.version)
print("License      :", dataset.metadata.license_)
print("Keywords     :", dataset.metadata.keywords)


## 2. Data Overview
Review available record sets, their fields, and associated `@id`s.

We'll list all record sets, their descriptions, and show the fields (columns) in each, referencing entities by their `@id`.

In [ ]:
# List all record sets with their @id and display their fields
record_sets = dataset.record_sets()
print(f"Found {len(record_sets)} record set(s):\n")

for idx, record_set in enumerate(record_sets):
    print(f"Record Set {idx+1}: @id={record_set.id}")
    print(f"  Name       : {record_set.name}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields (@id, name, dataType):")
    for field in record_set.fields:
        print(f"    - @id: {field.id}")
        print(f"      name    : {field.name}")
        print(f"      dataType: {field.data_type}")
        if hasattr(field, 'description'):
            print(f"      description: {field.description}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Note:** Use the record set and field `@id`s from the overview above. Here, as an example, we'll extract the first available record set.

*If your dataset has multiple record sets, you can repeat this cell for others as needed.*

In [ ]:
# Collect all record set @ids
record_set_ids = [rset.id for rset in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()
# Select the first record set for subsequent steps
main_record_set_id = record_set_ids[0]  # Update if another record set is desired
print(f"Using main_record_set_id: {main_record_set_id}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's explore this dataset with some example operations:
- Filtering records on a numeric field
- Normalizing that field
- Grouping records by a categorical field

We use field `@id` for reference. Please refer to the field list above to pick meaningful field IDs (columns) for your analysis.

In [ ]:
# Select a numeric field and a group field by their @id (as seen in overview section)
numeric_field_id = None
group_field_id = None

# Try to automatically select a numeric and categorical field if available
from pandas.api.types import is_numeric_dtype

df = dataframes[main_record_set_id]
for col in df.columns:
    if is_numeric_dtype(df[col]) and numeric_field_id is None:
        numeric_field_id = col
    if not is_numeric_dtype(df[col]) and group_field_id is None:
        group_field_id = col

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field:   {group_field_id}")

# Simple threshold for filtering
threshold = df[numeric_field_id].quantile(0.75)  # Top 25% as an example
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Top 5 normalized {numeric_field_id} values:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Grouping
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id + '_normalized'].mean().reset_index()
    print(f"Grouped (mean of normalized {numeric_field_id}) by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the numeric field and see mean values by group.

*Feel free to update the field IDs for more meaningful field names based on your actual data overview.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Boxplot by group (if categorical field exists and not too many groups)
if group_field_id is not None and df[group_field_id].nunique() < 30:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset using `mlcroissant`, reviewed its structure using record set and field `@id`s, extracted and explored the data, performed basic filtering and normalization, grouped data for summary statistics, and visualized key field distributions.

- For more detailed analysis, select the most relevant fields by their `@id` using the overview section above.
- Use `mlcroissant`'s rich metadata model for programmatic, reproducible data science workflows.

**Next steps:**
- Explore more fields or record sets
- Apply your own domain-specific filters
- Share your insights with collaborators!